# Лекция 3. Применение $\ell_1$-нормы и унитарные матрицы

## На прошлом занятии

- Форматы представления действительных чисел для обучения нейросетей
- Векторы и векторные нормы
- Введение в матричные нормы

## Спектральная норма

- Спектральная норма, $\Vert A \Vert_2$, одна из наиболее часто используемых, наряду с нормой Фробениуса
- Она не может быть вычислена напрямую через элементы матрицы, но для её вычисления существуют эффективные алгоритмы
- Она напрямую связана с сингулярным разложением матрицы (SVD) поскольку выполнено следующее равенство

$$
   \Vert A \Vert_2 = \sigma_1(A) = \sqrt{\lambda_\max(A^*A)}
$$

где $\sigma_1(A)$ – максимальное сингулярное число матрицы $A$. 

- Подробнее про SVD будет рассказано далее, а сейчас давайте вычислим все вышеупомянутые нормы

In [5]:
import numpy as np
n = 100
m = 100
a = np.random.randn(n, m) # Random n x m matrix
s1 = np.linalg.norm(a, 2) # Spectral
s2 = np.linalg.norm(a, 'fro') # Frobenius
s3 = np.linalg.norm(a, 1) # 1-norm
s4 = np.linalg.norm(a, np.inf) 
print('Spectral: {0:} \nFrobenius: {1:} \n1-norm: {2:} \ninfinity: {3:}'.format(s1, s2, s3, s4))

Spectral: 19.54312605521002 
Frobenius: 99.09062756034918 
1-norm: 94.67206933546186 
infinity: 94.90491842710985


## Почему важна $\ell_1$-норма?


$\ell_1$-норма играет важную роль в задаче compressed sensing.

Простейшая формулировка этой задачи следующая:

Даны некоторые наблюдения $b$
Известно, что модель получения наблюдений линейная $$Ax = b,$$
где 
$A$ – это известная $m \times n $ матрица и $m < n$.

**Q**: можем ли мы найти решение такой системы?

Решение, очевидно, не единственно, поэтому естественный подход – это искать решение минимальное в некотором смысле:

\begin{align*}
& \| x\| \to \min \\
\text{subject to } & Ax = b
\end{align*}
 
 

- Выбор стандартной евклидовой нормы $\| x\| = \|x\|_2$ приводит к линейной задаче наименьших квадратов

- Выбор первой нормы $\| x\| = \|x\|_1$ приводит к задаче [compressed sensing](https://en.wikipedia.org/wiki/Compressed_sensing)

- Обычно решение этой задачи является наиболее разреженным (sparse) решением данной системы

Basis pursuit - eщё одно название задачи
\begin{align*}
& \| x\|_1 \to \min \\
\text{subject to } & Ax = b
\end{align*}

- Формальная связь между разреженностью и $\ell_1$ нормой показана [тут](https://arxiv.org/pdf/math/0503066)

## Всегда ли использование $\ell_1$-нормы приводит к разреженности?

In [ ]:
import cvxpy as cp
import numpy as np
import matplotlib.pyplot as plt

n = 10
A = np.random.randn(n, 2*n)
# x_true = np.random.randn(2*n)
x_true = np.zeros(2*n)
x_true[1] = -4
x_true[10] = 3
eps = 1e-3
b = A @ x_true + eps * np.random.randn(n)

x = cp.Variable(2*n)
problem = cp.Problem(cp.Minimize(cp.norm(x, 1)), [A @ x == b])
problem.solve(verbose=True)
print(x.value)
plt.plot(abs(x.value))
plt.yscale("log")
x_approx = x.value.copy()
x_approx[np.abs(x_approx) < 1e-3] = 0
print(np.sum(x_approx != 0))
print(x_approx)
print(np.linalg.norm(A @ x_approx - b))

In [ ]:
import cvxpy as cp
import numpy as np

n = 10
A = np.zeros((n, n+2))
eps = 1e-6
A[:, 2:] = np.eye(n)
A[:, 0] = eps / 2
A[0, 0] += 1
A[1, 0] += -1
A[:, 1] = eps / 2
A[1, 1] += 1
A[0, 1] += -1
b = np.ones(n)
print(A)

x = cp.Variable(n+2)
problem = cp.Problem(cp.Minimize(cp.norm(x, 1)), [A @ x == b])
problem.solve()
print(x.value)
print(np.linalg.norm(x.value, 1))
x_sparse = np.zeros(n+2)
x_sparse[0] = 1 / eps
x_sparse[1] = 1 / eps
print(np.linalg.norm(A @ x_sparse - b))
print(np.linalg.norm(x_sparse, 1))
A[:, 1] @ b

## Скалярное произведение

Если норма помогает измерять расстояние, то **скалярное произведение** позволяет учесть угол между объектами.  

Скалярное произведение определено

* **для векторов:**
$$
   (x, y) =  x^{\top} y = \sum_{i=1}^n x_i y_i.
$$
Тогда евклидова норма записывается как
$$
   \Vert x \Vert_2 = \sqrt{(x, x)};
$$  


* **для матриц**:
$$
    (A, B)_F = \displaystyle{\sum_{i=1}^{n}\sum_{j=1}^{m}} a_{ij} b_{ij} \equiv \mathrm{trace}(A^{\top} B),
$$
где $\mathrm{trace}(A)$ обозначает след матрицы, то есть сумму диагональных элементов. 

**Упражнение:** покажите, что $\|A\|_F = \sqrt{(A, A)_F}$.

**Замечание**. Угол между векторами или матрицами определяется как

$$
   \cos \phi = \frac{(x, y)}{\Vert x \Vert_2 \Vert y \Vert_2}.
$$

## Матрицы, сохраняющие норму

- Для устойчивости вычислений необходимо, чтобы ошибка не возростала после применения некоторого преобразования. Найдём эти преобразования. 

- Пусть дан вектор $\widehat{x}$ – аппроксимация вектора $x$ такая что,  

$$
  \frac{\Vert x - \widehat{x} \Vert}{\Vert x \Vert} \leq \varepsilon.
$$

- Вычислим образ векторов $x$ и $\widehat{x}$ после применения линейного преобразования $U$:  

$$
   y = U x, \quad \widehat{y} = U \widehat{x}.
$$

- Для построения алгоритмов необходимо использовать преобразования, которые не увеличивают (или даже сохраняют) ошибку:

$$
   \frac{\Vert y - \widehat{y} \Vert}{\Vert y \Vert } = \frac{\Vert U ( x - \widehat{x}) \Vert}{\Vert U  x\Vert}  \leq \varepsilon.
$$

Вопрос состоит в том, какой класс матриц не меняет норму вектора после умножения матрицы из этого класса на вектор?

$$
\frac{\Vert U ( x - \widehat{x}) \Vert}{\Vert U  x\Vert} = \frac{ \|x - \widehat{x}\|}{\|x\|}.
$$

Для евклидовой нормы $\|\cdot\|_2$ таким свойством обладают **унитарные** (или ортогональные) матрицы.

## Унитарные (ортогональные) матрицы

Пусть $U$ комплексная $n \times n$ матрица, и $\Vert U z \Vert_2 = \Vert z \Vert_2$ для всех $z$. 

Это выполнено тогда и только тогда, когда

$$
   U^* U = I_n,
$$

где $I_n$ единичная матрица $n\times n$, и $[A^*]_{ij} = \bar{a}_{ji}$

Комплексная $n\times n$ квадратная матрица называется **унитарной** если

$$
    U^*U = UU^* = I_n,
$$

что означает, что столбцы и строки матрицы образуют базис в $\mathbb{C}^{n}$.

## Свойство ортогональных матриц

Произведение двух унитарных матриц – унитарная матрица:  

$$(UV)^* UV = V^* (U^* U) V = V^* V = I,$$

- Позже мы покажем, что существуют классы унитарных матриц, произведение которых может дать произвольную унитарную матрицу 
- Эта идея лежит в основе некоторых алгоритмов, например вычисления QR разложения

## Примеры  унитарных матриц

Два важных класса унитарных матриц, произведение которых может дать любую унитарную матрицу:
1. Матрицы Хаусхолдера
2. Матрицы Гивенса

Этот факт станет очевидным после того, как мы рассмотрим QR разложение и способы его вычисления.

Другие важные примеры
* **матрица перестановки** $P$ строки (столбцы) которой получены перестановкой строк (столбцов) единичной матрицы.
* **матрица Фурье** $F_n = \frac{1}{\sqrt{n}} \left\{ e^{-i\frac{2\pi kl}{n}}\right\}_{k,l=0}^{n-1}$

## Промежуточный итог

- Особенности $\ell_1$ нормы
- Унитарные (ортогональные) матрицы
- Примеры унитарных матриц